##### Copyright 2026 Google LLC.


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Data Parallel Inference with Gemma 3 on TPU v5e-8

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/tutorials/Data_Parallel_Inference_JAX_TPU_v5e8.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

*Author: [Ayush Debnath](https://github.com/ayushdebnath012)*

This notebook targets a **Kaggle TPU v5e-8** runtime, which exposes all eight
cores needed for the data-parallel mesh below. Colab's current single-chip
v5e-1/v6e-1 slices will run the code but cannot demonstrate multi-core sharding.


## Setup and Installation

First, install the required packages:

In [ ]:
!pip install -q -U keras>=3.0 keras-nlp jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

Import necessary libraries:

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import time
import jax
import keras
import keras_nlp
from keras.distribution import distribution as dist_lib
import numpy as np

print(f"JAX version: {jax.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"KerasNLP version: {keras_nlp.__version__}")
print(f"Backend: {keras.backend.backend()}")

## TPU Detection and Mesh Configuration

Verify TPU availability and configure the device mesh:

In [ ]:
# Detect TPU devices
devices = jax.devices()
print(f"Number of devices: {len(devices)}")
print(f"Device type: {devices[0].platform}")
print(f"\nAll devices:")
for i, device in enumerate(devices):
    print(f"  Device {i}: {device}")

# Verify you have 8 TPU cores
if len(devices) != 8:
    print(f"\n⚠️  Warning: Expected 8 TPU cores but found {len(devices)}")
    print("   Make sure you're running on Kaggle with TPU v5e-8 accelerator enabled.")
else:
    print(f"\n✓ Successfully detected 8 TPU cores!")

Create a device mesh for data parallelism. You'll use a simple 1D mesh where each of the 8 cores processes different data in parallel:

In [ ]:
# Create a 1D mesh with all 8 devices along the 'data' axis
mesh = jax.sharding.Mesh(devices, axis_names=('data',))

print(f"Device mesh shape: {mesh.shape}")
print(f"Axis names: {mesh.axis_names}")
print(f"\nMesh visualization:")
print(mesh)

## Configure Keras Distribution

Set up data parallelism using Keras Distribution API. This tells Keras how to distribute data and models across the TPU mesh:

In [ ]:
# Create a DataParallel distribution strategy
# This replicates the model on all devices and splits the batch across them
distribution = dist_lib.DataParallel(device_mesh=mesh)

# Set the distribution as the default
dist_lib.distribution.set_distribution(distribution)

print("Data parallel distribution configured!")
print(f"Model will be replicated across {len(devices)} devices")
print(f"Each device will process batch_size / {len(devices)} samples")

## Load Gemma 3 Model

Load the Gemma 3 270M model using Keras NLP. The model will automatically be distributed across TPU cores according to your distribution strategy:

In [ ]:
# Set Kaggle credentials (required for accessing Gemma models)
# These are automatically available in Kaggle notebooks
os.environ["KAGGLE_USERNAME"] = os.environ.get("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"] = os.environ.get("KAGGLE_KEY", "")

# Load Gemma 3 270M model
# The model is automatically distributed across all TPU cores
print("Loading Gemma 3 270M model...")
print("This may take a few minutes on first run (downloading weights)...\n")

model = keras_nlp.models.GemmaCausalLM.from_preset(
    "gemma_270m_en",  # Gemma 3 270M
    dtype="float16"    # Use float16 for faster inference on TPU
)

print("\n✓ Model loaded successfully!")
print(f"Model max sequence length: {model.preprocessor.sequence_length}")
print(f"Vocabulary size: {model.backbone.vocabulary_size}")

## Verify Model Sharding

Let's verify that the model is properly distributed across TPU cores:

In [ ]:
# Check how model weights are sharded
print("Model weight distribution:")
for i, weight in enumerate(model.weights[:3]):  # Show first 3 weights as example
    print(f"\nWeight {i}: {weight.name}")
    print(f"  Shape: {weight.shape}")
    print(f"  Dtype: {weight.dtype}")
    if hasattr(weight, 'sharding'):
        print(f"  Sharding: {weight.sharding}")

print(f"\n... ({len(model.weights) - 3} more weights)")

## Data Parallel Inference

Now you'll demonstrate data parallelism by processing multiple prompts simultaneously. Each TPU core will generate text for different prompts in parallel.

### Prepare Input Prompts

Create a batch of diverse prompts. With 8 TPU cores, you'll use a batch size of 8 (one prompt per core):

In [ ]:
# Create 8 diverse prompts for parallel processing
prompts = [
    "Write a haiku about artificial intelligence:",
    "Explain quantum computing in simple terms:",
    "List three benefits of renewable energy:",
    "Describe the water cycle:",
    "What is machine learning?",
    "Write a recipe for chocolate chip cookies:",
    "Explain photosynthesis:",
    "What are the planets in our solar system?"
]

print(f"Prepared {len(prompts)} prompts for data parallel inference")
print(f"\nPrompts:")
for i, prompt in enumerate(prompts):
    print(f"  {i+1}. {prompt}")

### Run Parallel Inference

Generate text for all prompts simultaneously using data parallelism:

In [ ]:
# Configure generation parameters
max_length = 128  # Maximum tokens to generate per prompt

print(f"Starting data parallel inference...")
print(f"  - Batch size: {len(prompts)}")
print(f"  - TPU cores: {len(devices)}")
print(f"  - Max generation length: {max_length} tokens\n")

# Time the inference
start_time = time.time()

# Generate text for all prompts in parallel
# Each TPU core processes one prompt
outputs = model.generate(prompts, max_length=max_length)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"\n✓ Inference completed in {elapsed_time:.2f} seconds")
print(f"  Average time per prompt: {elapsed_time / len(prompts):.2f} seconds")

### Display Results

Show the generated text for each prompt:

In [ ]:
print("\n" + "="*80)
print("GENERATED OUTPUTS")
print("="*80 + "\n")

for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    print(f"\n{'─'*80}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'─'*80}")
    print(output)

print(f"\n{'='*80}")

## Performance Comparison

Compare data parallel inference vs. sequential inference to demonstrate the speedup:

In [ ]:
# Benchmark sequential inference (one prompt at a time)
print("Running sequential inference benchmark...\n")

start_time = time.time()
sequential_outputs = []
for i, prompt in enumerate(prompts):
    print(f"Processing prompt {i+1}/{len(prompts)}...", end="\r")
    output = model.generate([prompt], max_length=max_length)
    sequential_outputs.append(output[0])
sequential_time = time.time() - start_time

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(f"\nData Parallel (8 cores):  {elapsed_time:.2f} seconds")
print(f"Sequential (single core): {sequential_time:.2f} seconds")
print(f"\nSpeedup: {sequential_time / elapsed_time:.2f}x")
print(f"Efficiency: {(sequential_time / elapsed_time) / len(devices) * 100:.1f}%")
print("\n" + "="*80)

## Advanced: Batch Size Scaling

Demonstrate how data parallelism scales with different batch sizes:

In [ ]:
# Test different batch sizes
batch_sizes = [8, 16, 24, 32]
test_prompt = "Write a short story about:"

print("Batch size scaling experiment:\n")
print(f"{'Batch Size':<12} {'Time (s)':<12} {'Tokens/sec':<15}")
print("─" * 40)

for batch_size in batch_sizes:
    # Create batch of prompts
    batch_prompts = [f"{test_prompt} topic {i}" for i in range(batch_size)]
    
    # Time the generation
    start = time.time()
    batch_outputs = model.generate(batch_prompts, max_length=64)
    duration = time.time() - start
    
    # Calculate throughput (approximate)
    total_tokens = batch_size * 64
    tokens_per_sec = total_tokens / duration
    
    print(f"{batch_size:<12} {duration:<12.2f} {tokens_per_sec:<15.1f}")

print("\nNote: Larger batches may show diminishing returns due to memory constraints.")

## Advanced: Custom Mesh Configurations

Explore different mesh topologies for various parallelism strategies:

In [ ]:
# Example 1: 1D mesh (data parallelism only)
mesh_1d = jax.sharding.Mesh(devices, axis_names=('data',))
print("1D Mesh (Pure Data Parallelism):")
print(f"  Shape: {mesh_1d.shape}")
print(f"  Axes: {mesh_1d.axis_names}")
print(f"  Use case: Maximum data throughput, small models\n")

# Example 2: 2D mesh (data + model parallelism)
# Reshape 8 devices into 2x4 (2 data replicas, 4-way model sharding)
devices_2d = np.array(devices).reshape(2, 4)
mesh_2d = jax.sharding.Mesh(devices_2d, axis_names=('data', 'model'))
print("2D Mesh (Data + Model Parallelism):")
print(f"  Shape: {mesh_2d.shape}")
print(f"  Axes: {mesh_2d.axis_names}")
print(f"  Use case: Larger models that don't fit on single device\n")

# Example 3: Alternative 2D mesh (4x2)
devices_2d_alt = np.array(devices).reshape(4, 2)
mesh_2d_alt = jax.sharding.Mesh(devices_2d_alt, axis_names=('data', 'model'))
print("Alternative 2D Mesh:")
print(f"  Shape: {mesh_2d_alt.shape}")
print(f"  Axes: {mesh_2d_alt.axis_names}")
print(f"  Use case: Balance between data throughput and model capacity\n")

print("Note: For Gemma 3 270M on v5e-8, pure data parallelism (1D mesh) is optimal.")
print("Larger Gemma models may benefit from 2D mesh configurations.")

## Memory and Resource Monitoring

Check TPU memory usage during inference:

In [ ]:
# Get memory statistics from JAX
try:
    memory_stats = jax.local_devices()[0].memory_stats()
    print("TPU Memory Statistics:")
    print(f"  Bytes in use: {memory_stats.get('bytes_in_use', 0) / 1e9:.2f} GB")
    print(f"  Peak bytes used: {memory_stats.get('peak_bytes_in_use', 0) / 1e9:.2f} GB")
    print(f"  Total bytes: {memory_stats.get('bytes_limit', 0) / 1e9:.2f} GB")
except Exception as e:
    print(f"Memory stats not available: {e}")
    print("Note: Detailed memory stats may not be available on all TPU versions.")

## Best Practices and Tips

### 1. Batch Size Selection
- Use batch sizes that are multiples of the number of TPU cores (8)
- Larger batches improve throughput but may hit memory limits
- For Gemma 3 270M on v5e-8, batches of 8-32 work well

### 2. Data Type Optimization
- Use `float16` or `bfloat16` for faster inference on TPU
- `float16` is generally sufficient for inference tasks
- Avoid `float32` unless precision is critical

### 3. Sequence Length Management
- Longer sequences require more memory per sample
- Reduce batch size if using longer sequences
- Gemma 3 supports up to 32k tokens, but memory scales linearly

### 4. Distribution Strategy
- **Data Parallelism**: Best for models that fit on single device (like Gemma 3 270M)
- **Model Parallelism**: Needed for larger models (Gemma 9B, 27B)
- **Pipeline Parallelism**: For very large models with sequential stages

### 5. Kaggle-Specific Tips
- Enable TPU v5e-8 accelerator in notebook settings
- Accept Gemma model terms at Kaggle Models
- First run downloads model weights (~1GB for 270M)
- Subsequent runs are much faster (weights cached)

### 6. Performance Optimization
- JIT compile functions for repeated use
- Avoid Python loops; use vectorized operations
- Minimize data transfer between host and device
- Use asynchronous execution when possible

## Troubleshooting

### Common Issues:

**1. "Expected 8 TPU cores but found X"**
- Make sure you're running on Kaggle (not Colab)
- Enable TPU v5e-8 accelerator in notebook settings
- Restart the notebook if TPU detection fails

**2. "Out of Memory" errors**
- Reduce batch size
- Decrease max_length for generation
- Use float16 instead of float32

**3. Model access errors**
- Accept Gemma model terms at [Kaggle Models](https://www.kaggle.com/models/google/gemma-2)
- Verify Kaggle credentials are set correctly
- Check internet connection for model download

**4. Slow first run**
- First run downloads model weights (~1GB)
- Subsequent runs use cached weights and are much faster
- XLA compilation happens on first inference (normal)

**5. Distribution not working**
- Verify distribution is set before loading model
- Check mesh configuration matches device count
- Ensure Keras backend is set to JAX

## Next Steps and Resources

### Try These Exercises:

1. **Experiment with longer contexts**: Test Gemma 3's 32k context window with longer prompts
2. **Try different mesh configurations**: Implement 2D mesh for model parallelism
3. **Fine-tune for your task**: Use data parallelism for faster fine-tuning
4. **Benchmark larger models**: Try Gemma 2B or 9B with mixed parallelism strategies

### Additional Resources:

- [Keras Documentation](https://keras.io/keras_3/)
- [Keras Distribution API Guide](https://keras.io/guides/distribution/)
- [JAX Documentation](https://jax.readthedocs.io/)
- [Gemma Models on Kaggle](https://www.kaggle.com/models/google/gemma-2)
- [TPU Best Practices](https://cloud.google.com/tpu/docs/performance-guide)

### Related Notebooks:

- Gemma Fine-tuning with Data Parallelism
- Gemma Model Parallelism for Large Models
- Advanced JAX Optimization Techniques

## Conclusion

This notebook demonstrated modern data parallelism for Gemma 3 using:

✅ **Keras 3** with JAX backend for built-in TPU support  
✅ **Keras Distribution API** for clean, maintainable parallelism code  
✅ **Kaggle TPU v5e-8** for accessible multi-core experimentation  
✅ **Future-proof stack** compatible with latest JAX/Keras ecosystem  

Key takeaways:
- Data parallelism provides near-linear speedup for inference workloads
- Keras Distribution API simplifies distributed computing
- TPU v5e-8 on Kaggle offers excellent price/performance for learning
- Modern JAX/Keras stack is more maintainable than legacy approaches

### Feedback

Questions or suggestions? Open an issue on the [Gemma Cookbook GitHub](https://github.com/google-gemma/cookbook).

---

*Notebook created for issue #275: Modernize JAX/TPU Parallelism with Gemma 3*